<div style="background-color:#EAEAEA; padding:20px; border-left:5px solid #6C757D; border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;"><td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1><hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de València</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px; font-weight:700; line-height:1.3; margin-top:14px; margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan — <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente — <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none; width:230px; padding-left:25px; text-align:right; vertical-align:top;"><img src="../../../../logo_uv.png" alt="Universitat de València" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# SNO Canonical Data Generator

Downloads the official pure-D2O release and stores detector observations separately from solar production inputs.

## 1. Imports and paths

In [1]:
from pathlib import Path
from io import BytesIO, StringIO
import hashlib
import json
import re
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the TPeanuts project root")


def fetch(url: str, *, timeout: int = 60) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": "TPeanuts/solar-data"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def download(url: str, path: Path, *, overwrite: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite or not path.exists():
        path.write_bytes(fetch(url))
    return path


PROJECT_ROOT = find_project_root(Path.cwd())
SOLAR_DATA = PROJECT_ROOT / "data" / "solar"

## 2. Download and convert

In [2]:
PROVIDER_DIR = SOLAR_DATA / "sno"
for category in ("raw", "observation", "metadata"):
    (PROVIDER_DIR / category).mkdir(parents=True, exist_ok=True)
BASE = "https://sno.phy.queensu.ca/sno/prlwebpage"
URLS = {"SnoDayNightPrlSpectra.dat": f"{BASE}/SnoDayNightPrlSpectra.dat", "SnoCosZenith.dat": f"{BASE}/SnoCosZenith.dat", "SnoBackgroundTables.dat": f"{BASE}/SnoBackgroundTables.dat", "prldata.ps": f"{BASE}/prldata.ps"}
paths = {name: download(url, PROVIDER_DIR / "raw" / name) for name, url in URLS.items()}

def numeric_lines(path, minimum):
    rows = []
    for line in path.read_text(errors="replace").splitlines():
        try:
            values = [float(token) for token in line.split()]
        except ValueError:
            continue
        if len(values) >= minimum:
            rows.append(values)
    return rows

spectra_rows = numeric_lines(paths["SnoDayNightPrlSpectra.dat"], 5)
pd.DataFrame(spectra_rows, columns=["bin", "energy_low_MeV", "energy_high_MeV", "day_counts", "night_counts"]).to_csv(PROVIDER_DIR / "observation" / "day_night_spectrum.csv", index=False)
zenith_values = [value for row in numeric_lines(paths["SnoCosZenith.dat"], 1) for value in row]
cos_zenith = -1.0 + (np.arange(len(zenith_values)) + 0.5) * 2.0 / len(zenith_values)
pd.DataFrame({"cos_zenith": cos_zenith, "livetime_s": zenith_values}).to_csv(PROVIDER_DIR / "observation" / "coszenith_exposure.csv", index=False)
background_rows = numeric_lines(paths["SnoBackgroundTables.dat"], 11)
background_columns = ["bin", "energy_low_MeV", "energy_high_MeV", "day_neutron", "day_neutron_error", "day_low_energy", "day_low_energy_error", "night_neutron", "night_neutron_error", "night_low_energy", "night_low_energy_error"]
pd.DataFrame(background_rows, columns=background_columns).to_csv(PROVIDER_DIR / "observation" / "backgrounds.csv", index=False)
(PROVIDER_DIR / "metadata" / "source.json").write_text(json.dumps({"provider": "SNO", "url": "https://sno.phy.queensu.ca/sno/prlwebpage/", "products": ["detector observations", "zenith exposure"], "not_production_spectra": True}, indent=2) + "\n")
print("SNO detector-level observations generated. No solar-model density or production spectrum was fabricated.")

SNO detector-level observations generated. No solar-model density or production spectrum was fabricated.


## 3. Validation

In [3]:
assert all(path.stat().st_size > 0 for path in paths.values())
assert len(spectra_rows) == 17 and len(zenith_values) == 480 and len(background_rows) == 17
print("SNO public-release products validated.")

SNO public-release products validated.
